# Sneak BAND handoff: SNEAK-013

Experiment log and custody companion. Jerry's official reply check and the Tom→Jerry handoff passed. A real question was posted to Byron; no human answer is asserted. These are captured API records, not independent provider attestations.

Inputs: `receipts/band/SNEAK-013_HANDOFF.json`. Python standard library and Node.js are sufficient for the calculations. No API keys or live provider calls are required. This verifies recorded evidence, not current service health.


In [1]:
from pathlib import Path
import json, hashlib, subprocess
ROOT = Path.cwd()
if not (ROOT/'receipts/band/SNEAK-013_HANDOFF.json').exists(): ROOT = ROOT.parent
receipt = json.loads((ROOT/'receipts/band/SNEAK-013_HANDOFF.json').read_text())
events = {x['event']['kind']: x['event']['payload'] for x in receipt['leaves']}
assert events['official_real_reply']['exit'] == 0
assert events['agent_handoff']['observed_outcome'] == 'PASS'
assert events['human_request']['observed_outcome'] == 'REQUEST_OBSERVED'
assert events['human_request']['human_response'] is None
print({'jerry_reply':'PASS','direct_handoff':'PASS','human_question':'REQUEST_OBSERVED','human_decision':'PENDING'})


{'jerry_reply': 'PASS', 'direct_handoff': 'PASS', 'human_question': 'REQUEST_OBSERVED', 'human_decision': 'PENDING'}


In [2]:
def h(b): return hashlib.sha256(b).digest()
def canonical(x): return json.dumps(x, sort_keys=True, ensure_ascii=True, separators=(',', ':')).encode('ascii')
def parent(a,b): return h(b'\x01'+a+b)
def merkle(xs):
    assert xs
    while len(xs)>1: xs=[parent(xs[i],xs[i+1]) if i+1<len(xs) else xs[i] for i in range(0,len(xs),2)]
    return xs[0]
def verify(leaf, proof, root):
    value=leaf
    for item in proof:
        if item['side'] not in ('left','right'): return False
        sibling=bytes.fromhex(item['hash'])
        if len(sibling)!=32: return False
        value=parent(sibling,value) if item['side']=='left' else parent(value,sibling)
    return value==root
leaves=[]
for item in receipt['leaves']:
    b=canonical(item['event']); assert b.hex()==item['canonical_hex']
    leaf=h(b'\x00'+b); assert leaf.hex()==item['leaf']; leaves.append(leaf)
root=merkle(leaves)
assert root.hex()==receipt['root']
assert all(verify(x,item['proof'],root) for x,item in zip(leaves,receipt['leaves']))
assert not verify(h(b'tampered'),receipt['leaves'][0]['proof'],root)
assert not verify(leaves[0],receipt['leaves'][0]['proof'],bytes(32))
assert merkle(leaves[:-1])!=root
print({'python':'PASS','root':root.hex(),'leaves':len(leaves),'tampered_leaf_accepted':False,'wrong_root_accepted':False,'removal_preserves_root':False})


{'python': 'PASS', 'root': '29002d5e05b18b58dceec7d1492a1300efc9bbf7d33e0fdafed755f43561895f', 'leaves': 3, 'tampered_leaf_accepted': False, 'wrong_root_accepted': False, 'removal_preserves_root': False}


In [3]:
NODE = "const fs=require(\"node:fs\"),crypto=require(\"node:crypto\"),assert=require(\"node:assert/strict\");\nconst r=JSON.parse(fs.readFileSync(0,\"utf8\"));\nconst sort=x=>Array.isArray(x)?x.map(sort):x&&typeof x===\"object\"?Object.fromEntries(Object.keys(x).sort().map(k=>[k,sort(x[k])])):x;\nconst canon=x=>JSON.stringify(sort(x)).replace(/[\\u007f-\\uffff]/g,c=>\"\\\\u\"+c.charCodeAt(0).toString(16).padStart(4,\"0\"));\nconst h=b=>crypto.createHash(\"sha256\").update(b).digest();\nconst par=(a,b)=>h(Buffer.concat([Buffer.from([1]),a,b]));\nconst ids=r.leaves.map(x=>x.event.id);assert.deepEqual(ids,[...ids].sort());assert.equal(r.leaf_count,ids.length);\nlet xs=r.leaves.map(item=>{\n const b=Buffer.from(canon(item.event),\"ascii\");assert.equal(b.toString(\"hex\"),item.canonical_hex);\n const leaf=h(Buffer.concat([Buffer.from([0]),b]));assert.equal(leaf.toString(\"hex\"),item.leaf);\n let v=leaf;\n for(const p of item.proof){assert.ok([\"left\",\"right\"].includes(p.side));assert.match(p.hash,/^[0-9a-f]{64}$/);const s=Buffer.from(p.hash,\"hex\");v=p.side===\"left\"?par(s,v):par(v,s);}\n assert.equal(v.toString(\"hex\"),r.root);return leaf;\n});\nwhile(xs.length>1){const n=[];for(let i=0;i<xs.length;i+=2)n.push(i+1<xs.length?par(xs[i],xs[i+1]):xs[i]);xs=n;}\nassert.equal(xs[0].toString(\"hex\"),r.root);\nconsole.log(JSON.stringify({independent_node:\"PASS\",root:r.root,leaves:r.leaf_count}));"
p = subprocess.run(['node','-e',NODE],input=json.dumps(receipt),text=True,capture_output=True)
assert p.returncode == 0, p.stderr
print(p.stdout)


{"independent_node":"PASS","root":"29002d5e05b18b58dceec7d1492a1300efc9bbf7d33e0fdafed755f43561895f","leaves":3}



## Limits and next action

Merkle membership establishes integrity of the declared captured records, not authorship, provider truth, causality, human approval, or complete application acceptance. No MMR or signature was created. Both agents use the Claude SDK; mixed-harness collaboration, BAND Desktop visibility, Cognee, Snyk, Flexprice, sandboxed red-team effects and LOMO interventions remain NOT_TESTED. Byron must answer SNEAK-H01 in the BAND room; later evidence must be a successor.
